# Rurality and population density → ITL2 (2025 and 2021 boundaries)

Two complementary measures of settlement structure, built for **both ITL2 2025** (46 regions → `clean/`) **and ITL2 2021** (41 regions → `clean/itl2_2021/`):

1. **Rurality (UK, one Census snapshot per nation)** — `rurality_itl2.csv`. Each UK nation has its own official classification, so the rural definition differs by nation (`rural_definition` column):
   - **England & Wales** — ONS *Rural Urban Classification 2021*, Table 1E: % of each LA's population in **rural Output Areas** (settlements < 10,000), and % in OAs **further from a major town or city**; × Census 2021 LA population (Nomis TS001).
   - **Scotland** — Scottish Government *Urban Rural Classification 2022*, Census 2022 population tables (CA6FOLD): % of each council area's population in each 6-fold class. Rural = *Accessible rural* + *Remote rural* (settlements < 3,000); `share_remote` = *Remote small towns* + *Remote rural* (> 30 min drive from a settlement of 10,000+); × Census 2022 council population (sum of 2022 Data Zone populations).
   - **Northern Ireland** — NISRA *Settlement 2015* classification, default urban/rural (rural = bands F–H, settlements < 5,000), Census **2011** populations (the classification's own tabulation). NI = one ITL2 region (TLN0).
   - Counts first: rural residents per LA (council / NI) → **summed to ITL2**; shares derived at ITL2.
   - **⚠ Not directly comparable across nations** (different thresholds and years): compare within nations, or use a nation fixed effect / the population-weighted density measure for UK-wide comparisons.
2. **Population density (UK-wide, annual 1991–2025)** — `population_density_itl2.csv`
   - Land area: ONS *Standard Area Measurements* for LAs (December 2024), **land only** (excludes inland water), summed to ITL2.
   - Population: the same Nomis LA mid-year estimates as `clean/population_itl2.csv` (checked identical).

**Caveats:** rurality is a single 2021 cross-section (use as a time-invariant control). LAs split across ITL2 (North Ayrshire; Argyll & Bute on the 2021 map) are assigned wholly as in the spines, so their land area goes with them — a small misallocation for Arran/Cumbrae and Helensburgh & Lomond (Scottish regions; affects density only).

All source URLs are defined below.

In [1]:
import io, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
RUC_FILE  = BASE / "raw" / "demographic" / "rural-urban-classification-2021.xlsx"
POP_LA    = sorted((BASE / "raw" / "population").glob("nomis_population_raw_*.csv"))[-1]     # from build/population.py
CROSSWALK = BASE / "lib" / "la_code_crosswalk.csv"
RAW_DIR   = BASE / "raw" / "geography"
SPINES = {"2025": (BASE / "clean" / "geography_spine.csv",             BASE / "clean"),
          "2021": (BASE / "clean" / "itl2_2021" / "geography_spine.csv", BASE / "clean" / "itl2_2021")}

URL_SAM   = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
             "Standard_Area_Measurements_for_the_Local_Authority_Districts_(December_2024)_in_the_UK/"
             "FeatureServer/0/query?where=1%3D1&outFields=LAD24CD,LAD24NM,AREALHECT&returnGeometry=false&f=json")
URL_TS001 = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_2021_1.data.csv?geography=TYPE424"
             "&c2021_restype_3=0&measures=20100&select=geography_code,obs_value")      # Census 2021 usual residents, LAs
print(f"Land area (ONS SAM, LAD Dec 2024):\n  {URL_SAM}\nCensus 2021 population (TS001):\n  {URL_TS001}\nRUC 2021 tables: {RUC_FILE.name}\nLA population series: {POP_LA.name}")

SCOT_UR   = BASE / "raw" / "density" / "sgur2022_population_tables.xlsx"      # downloaded by build/density.ipynb
SCOT_DZ   = BASE / "raw" / "density" / "SG_DataZoneBdry_2022.zip"
SCOT_SAPE = BASE / "raw" / "density" / "nrs_sape_2011_2024_dz2022.zip"
NI_SETTLE = BASE / "raw" / "density" / "nisra_Settlement15-lookup.xls"
print("Scotland URC 2022 tables: https://www.gov.scot/publications/scottish-government-urban-rural-classification-2022/")
print("NI Settlement 2015 lookup: https://www.nisra.gov.uk/files/nisra/publications/Settlement15-lookup.xls")

remap = dict(zip(*[pd.read_csv(CROSSWALK, dtype=str)[c] for c in ("old_code", "new_code")]))
def current(codes):
    return codes.astype(str).str.strip().replace(remap)

Land area (ONS SAM, LAD Dec 2024):
  https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Standard_Area_Measurements_for_the_Local_Authority_Districts_(December_2024)_in_the_UK/FeatureServer/0/query?where=1%3D1&outFields=LAD24CD,LAD24NM,AREALHECT&returnGeometry=false&f=json
Census 2021 population (TS001):
  https://www.nomisweb.co.uk/api/v01/dataset/NM_2021_1.data.csv?geography=TYPE424&c2021_restype_3=0&measures=20100&select=geography_code,obs_value
RUC 2021 tables: rural-urban-classification-2021.xlsx
LA population series: nomis_population_raw_2026-10-07.csv
Scotland URC 2022 tables: https://www.gov.scot/publications/scottish-government-urban-rural-classification-2022/
NI Settlement 2015 lookup: https://www.nisra.gov.uk/files/nisra/publications/Settlement15-lookup.xls


## 1. LA-level inputs

RUC Table 1E (% rural, % further from a major town), Census 2021 LA population, and LA land area. Old LA codes are mapped to current codes with the crosswalk before anything is joined.

In [2]:
ruc = pd.read_excel(RUC_FILE, sheet_name="Table 1E", header=2)
ruc = ruc.rename(columns={"LAD24CD": "la_code", "Proportion of population in rural OAs (%)": "pct_rural",
                          "Proportion of population in OAs further from a major town or city (%)": "pct_further"})
ruc["la_code"] = current(ruc.la_code)
ruc = ruc[["la_code", "RUC21CD", "RUC21NM", "pct_rural", "pct_further"]].dropna(subset=["pct_rural"])

cen = pd.read_csv(URL_TS001)
cen = cen.assign(la_code=current(cen.GEOGRAPHY_CODE)).groupby("la_code").OBS_VALUE.sum().rename("pop_census2021")
ruc = ruc.join(cen, on="la_code")
assert ruc.pop_census2021.notna().all(), f"no Census population for: {ruc.loc[ruc.pop_census2021.isna(), 'la_code'].tolist()}"
ruc["pop_rural"]   = ruc.pct_rural   / 100 * ruc.pop_census2021
ruc["pop_further"] = ruc.pct_further / 100 * ruc.pop_census2021
print(f"RUC: {len(ruc)} E&W LAs | rural residents {ruc.pop_rural.sum():,.0f} of {ruc.pop_census2021.sum():,.0f} "
      f"({100 * ruc.pop_rural.sum() / ruc.pop_census2021.sum():.1f}%)")

# check against ONS's own national totals (Table 2A: population by OA classification)
t2a = pd.read_excel(RUC_FILE, sheet_name="Table 2A", header=2)
rural_ons = t2a.loc[t2a.RUC21NM.str.startswith(("Larger rural", "Smaller rural")), "Population"].sum()
further_ons = t2a.loc[t2a.RUC21NM.str.contains("Further from"), "Population"].sum()
print(f"Check vs ONS national totals: rural {100 * (ruc.pop_rural.sum() / rural_ons - 1):+.2f}%, "
      f"further from major town {100 * (ruc.pop_further.sum() / further_ons - 1):+.2f}%  (LA % rounded to 0.1)")

ruc["rural_definition"] = "E&W RUC 2021: rural OAs (settlements <10k), Census 2021"

# --- Scotland: URC 2022 6-fold shares by council area x Census 2022 council population ---
import geopandas as gpd, zipfile
ur = pd.read_excel(SCOT_UR, sheet_name="CA6FOLD", header=2).dropna(subset=["CACode"])
dz = gpd.read_file(f"zip://{SCOT_DZ}!SG_DataZone_Bdry_2022.shp", ignore_geometry=True)
zs = zipfile.ZipFile(SCOT_SAPE)
dz_ca = pd.read_excel(zs.open(next(n for n in zs.namelist() if "2022" in n)), sheet_name="2022", header=2, usecols=[0, 2])
dz_ca.columns = ["dzcode", "la_code"]
ca_pop = dz.merge(dz_ca.drop_duplicates(), on="dzcode").groupby("la_code").totpop2022.sum()
scot = pd.DataFrame({"la_code": ur.CACode, "pop_census2021": ur.CACode.map(ca_pop)})      # census population base (2022 for Scotland)
assert scot.pop_census2021.notna().all()
scot["pop_rural"]  = (ur["Accessible rural"] + ur["Remote rural"]).values * scot.pop_census2021
scot["pop_remote"] = (ur["Remote small towns"] + ur["Remote rural"]).values * scot.pop_census2021
scot["rural_definition"] = "Scotland URC 2022: accessible + remote rural (settlements <3k), Census 2022"
print(f"Scotland: {len(scot)} councils | rural {100 * scot.pop_rural.sum() / scot.pop_census2021.sum():.1f}% "
      f"| remote {100 * scot.pop_remote.sum() / scot.pop_census2021.sum():.1f}% of {scot.pop_census2021.sum():,.0f}")

# --- Northern Ireland: Settlement 2015 default urban/rural, 2011 Census (NI = ITL2 TLN0) ---
q = pd.read_excel(NI_SETTLE, sheet_name="Quality - Population", header=3)
band = q["SETTLEMENT 2015 BAND"].astype(str).str.strip()
q["Exact Counts"] = pd.to_numeric(q["Exact Counts"], errors="coerce")
ni_urban = q.loc[band == "A-E", "Exact Counts"].sum(); ni_rural = q.loc[band == "F-H", "Exact Counts"].sum()
ni = pd.DataFrame({"la_code": ["N92000002"], "pop_census2021": [ni_urban + ni_rural], "pop_rural": [ni_rural],
                   "rural_definition": ["NI Settlement 2015: bands F-H (settlements <5k), Census 2011"]})
print(f"NI: rural {100 * ni_rural / (ni_urban + ni_rural):.1f}% of {ni_urban + ni_rural:,.0f} (Census 2011)")

rural_la = pd.concat([ruc, scot, ni], ignore_index=True)

sam = pd.DataFrame([f["attributes"] for f in json.loads(urllib.request.urlopen(URL_SAM, timeout=120).read())["features"]])
sam.to_csv(RAW_DIR / "SAM_LAD_Dec2024_UK.csv", index=False)
area = sam.assign(la_code=current(sam.LAD24CD)).groupby("la_code").AREALHECT.sum().div(100).rename("land_area_km2")
print(f"Land area: {len(area)} UK LAs, {area.sum():,.0f} km2")

pop = pd.read_csv(POP_LA)
pop = pop.assign(la_code=current(pop.GEOGRAPHY_CODE), year=pop.DATE).groupby(["la_code", "year"]).OBS_VALUE.sum().rename("population").reset_index()

RUC: 318 E&W LAs | rural residents 10,458,871 of 59,597,542 (17.5%)


Check vs ONS national totals: rural -0.01%, further from major town -0.01%  (LA % rounded to 0.1)


Scotland: 32 councils | rural 16.7% | remote 6.5% of 5,439,565
NI: rural 37.5% of 1,810,863 (Census 2011)


Land area: 361 UK LAs, 242,743 km2


## 2. Sum to ITL2 (both boundary versions) and save

Rural shares are computed at ITL2 from the summed counts. Scottish and NI regions have no RUC (NaN). Density = population ÷ land area.

In [3]:
for vintage, (spine_path, out_dir) in SPINES.items():
    spine = pd.read_csv(spine_path)
    assert spine.la_code.is_unique
    to_itl2 = dict(zip(spine.la_code, spine.itl2_code)); names = dict(zip(spine.itl2_code, spine.itl2_name))
    to_itl2["N92000002"] = "TLN0"                                  # NI rural figure is for NI as a whole
    for name, d in [("rurality", rural_la), ("land area", area.reset_index()), ("population", pop)]:
        bad = sorted(set(d.la_code) - set(to_itl2))
        assert not bad, f"{name}: LA codes not in the {vintage} spine: {bad[:10]}"
    assert set(spine.la_code) <= set(area.index), "LA without land area"

    rl = rural_la.assign(itl2_code=rural_la.la_code.map(to_itl2))
    r = rl.groupby("itl2_code")[["pop_census2021", "pop_rural", "pop_further", "pop_remote"]].sum(min_count=1)
    r["share_rural"] = 100 * r.pop_rural / r.pop_census2021
    r["share_further_from_major_town"] = 100 * r.pop_further / r.pop_census2021     # E&W only
    r["share_remote"] = 100 * r.pop_remote / r.pop_census2021                       # Scotland only
    r["rural_definition"] = rl.groupby("itl2_code").rural_definition.first()
    r = r.rename(columns={"pop_census2021": "pop_census"})
    la_area = area.reset_index().assign(itl2_code=lambda d: d.la_code.map(to_itl2)).groupby("itl2_code").land_area_km2.sum()
    rur = (pd.DataFrame(index=sorted(names)).rename_axis("itl2_code").join(r).join(la_area).reset_index())
    rur.insert(1, "itl2_name", rur.itl2_code.map(names))
    rur.to_csv(out_dir / "rurality_itl2.csv", index=False, float_format="%.4f")

    dens = pop.assign(itl2_code=pop.la_code.map(to_itl2)).groupby(["itl2_code", "year"]).population.sum().reset_index()
    dens = dens.merge(la_area.reset_index(), on="itl2_code")
    dens["pop_per_km2"] = dens.population / dens.land_area_km2
    dens.insert(1, "itl2_name", dens.itl2_code.map(names))
    dens = dens.sort_values(["year", "itl2_code"]).reset_index(drop=True)
    dens.to_csv(out_dir / "population_density_itl2.csv", index=False, float_format="%.4f")

    print(f"\nITL2 {vintage}: {len(rur)} regions | rurality for {rur.pop_census.notna().sum()} | "
          f"density {dens.itl2_code.nunique()} regions x {dens.year.nunique()} years -> {out_dir.name}/")
    if vintage == "2025":
        p = pd.read_csv(BASE / "clean" / "population_itl2.csv").merge(dens, on=["itl2_code", "year"], suffixes=("_ref", ""))
        print(f"  population matches clean/population_itl2.csv: max |gap| {(p.population - p.population_ref).abs().max():.0f}")
    for nat, pre in [("E&W", ("TLC","TLD","TLE","TLF","TLG","TLH","TLI","TLJ","TLK","TLL")), ("Scotland", ("TLM",)), ("NI", ("TLN",))]:
        show = rur[rur.itl2_code.str.startswith(pre)].sort_values("share_rural", ascending=False)
        print(f"  {nat}: " + ", ".join(f"{n} {s:.0f}%" for n, s in show[["itl2_name", "share_rural"]].head(3).values))


ITL2 2025: 46 regions | rurality for 46 | density 46 regions x 35 years -> clean/
  population matches clean/population_itl2.csv: max |gap| 0
  E&W: Cornwall and Isles of Scilly 56%, Cumbria 50%, Norfolk 49%
  Scotland: Highlands and Islands 50%, North Eastern Scotland 28%, Southern Scotland 23%
  NI: Northern Ireland 37%

ITL2 2021: 41 regions | rurality for 41 | density 41 regions x 35 years -> itl2_2021/
  E&W: Cornwall and Isles of Scilly 56%, Cumbria 50%, Lincolnshire 47%
  Scotland: Highlands and Islands 50%, North Eastern Scotland 28%, Southern Scotland 23%
  NI: Northern Ireland 37%
